In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc

## Read in CGPS+GMIMS Q/U files, CGPS only Q/U files and smoothed GMIMS Q/U files 

In [ ]:
CG_Q_list = []
CG_U_list = []
G_Q_list = []
G_U_list = []
C_Q_list = []
C_U_list = []

band = ['A','B','C','D']
bandlc = ['a','b','c','d']
for i in range(0,4):
    
    print('band '+band[i])
    
    # CGPS + GMIMS (CG)
    hdu_CG_Q = fits.open('/home2/DATA_AO/CGPS_GMIMS/Q'+band[i]+'_CG.fits')
    hdu_CG_U = fits.open('/home2/DATA_AO/CGPS_GMIMS/U'+band[i]+'_CG.fits')
    CG_Q_list.append(hdu_CG_Q[0].data)
    CG_U_list.append(hdu_CG_U[0].data)
    
    # Grab a header (all the same for now)
    hdr = hdu_CG_Q[0].header
    
    # CGPS only (C)
    hdu_C_Q = fits.open('/home2/DATA_AO/CGPS_GMIMS/Q'+band[i]+'_C.fits')
    hdu_C_U = fits.open('/home2/DATA_AO/CGPS_GMIMS/U'+band[i]+'_C.fits')
    C_Q_list.append(hdu_C_Q[0].data)
    C_U_list.append(hdu_C_U[0].data)
    hdr = hdu_C_Q[0].header
    
    # GMIMS only (G)
    hdu_G_Q = fits.open('/home/ordoga/DRAO_export/CG_W23/GMIMS_'+bandlc[i]+'/Q'+band[i]+'_GMIMS_W21_HR.fits')
    hdu_G_U = fits.open('/home/ordoga/DRAO_export/CG_W23/GMIMS_'+bandlc[i]+'/U'+band[i]+'_GMIMS_W21_HR.fits')   
    G_Q_reproj, footprint = reproject_interp(hdu_G_Q,hdr)
    G_U_reproj, footprint = reproject_interp(hdu_G_U,hdr)    
    G_Q_list.append(G_Q_reproj)
    G_U_list.append(G_U_reproj)

del G_Q_reproj, G_U_reproj, footprint
gc.collect()

In [ ]:
map_QU_cubes = True

if map_QU_cubes:

    #==============
    band = 'B'
    #==============

    wcs = WCS(hdr)
    fs = 36

    if band == 'A': i = 0
    if band == 'B': i = 2
    if band == 'C': i = 4
    if band == 'D': i = 6        

    plt.figure(figsize=(80,8))
    plt.subplot(projection=wcs)
    plt.imshow(G_Q_list[i],origin='lower',vmin=-0.5,vmax=0.5,cmap='RdBu_r')
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs) 

    plt.figure(figsize=(80,8))
    plt.subplot(projection=wcs)
    plt.imshow(G_U_list[i+1],origin='lower',vmin=-0.5,vmax=0.5,cmap='RdBu_r')
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs) 
    
    plt.figure(figsize=(80,8))
    plt.subplot(projection=wcs)
    plt.imshow(np.sqrt(G_U_list[i+1]**2+G_Q_list[i]**2),origin='lower',vmin=0.0,vmax=1.0,cmap='viridis')
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs) 

## Make GMIMS Q and U (smoothed) files with new projection

In [ ]:
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QA_G.fits',G_Q_list[0],header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QB_G.fits',G_Q_list[1],header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QC_G.fits',G_Q_list[2],header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QD_G.fits',G_Q_list[3],header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UA_G.fits',G_U_list[0],header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UB_G.fits',G_U_list[1],header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UC_G.fits',G_U_list[2],header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UD_G.fits',G_U_list[3],header=hdr,overwrite=True)

## Make PI for CGPS, GMIMS, CGPS+GMIMS (not convolved)

In [ ]:
PI_C = np.nanmean(np.sqrt(np.array(C_Q_list)**2+np.array(C_U_list)**2),axis=0)
PI_G = np.nanmean(np.sqrt(np.array(G_Q_list)**2+np.array(G_U_list)**2),axis=0)
PI_CG = np.nanmean(np.sqrt(np.array(CG_Q_list)**2+np.array(CG_U_list)**2),axis=0)

fits.writeto('/home2/DATA_AO/CGPS_GMIMS/PI_C.fits',PI_C,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/PI_G.fits',PI_G,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/PI_CG.fits',PI_CG,header=hdr,overwrite=True)

del PI_C
del PI_G
del PI_CG
gc.collect()

## Convolve Q,U with Gaussian beam

In [ ]:
kernel = Gaussian2DKernel(x_stddev=2,y_stddev=2)
print('QA')
QA_GC_conv = convolve(CG_Q_list[0], kernel)
QA_C_conv = convolve(C_Q_list[0], kernel)
print('QB')
QB_GC_conv = convolve(CG_Q_list[1], kernel)
QB_C_conv = convolve(C_Q_list[1], kernel)
print('QC')
QC_GC_conv = convolve(CG_Q_list[2], kernel)
QC_C_conv = convolve(C_Q_list[2], kernel)
print('QD')
QD_GC_conv = convolve(CG_Q_list[3], kernel)
QD_C_conv = convolve(C_Q_list[3], kernel)
print('UA')
UA_GC_conv = convolve(CG_U_list[0], kernel)
UA_C_conv = convolve(C_U_list[0], kernel)
print('UB')
UB_GC_conv = convolve(CG_U_list[1], kernel)
UB_C_conv = convolve(C_U_list[1], kernel)
print('UC')
UC_GC_conv = convolve(CG_U_list[2], kernel)
UC_C_conv = convolve(C_U_list[2], kernel)
print('UD')
UD_GC_conv = convolve(CG_U_list[3], kernel)
UD_C_conv = convolve(C_U_list[3], kernel)

In [ ]:
del CG_Q_list
del CG_U_list
del C_Q_list
del C_U_list
gc.collect()

## Write out convolved CGSP,CGPS+GMIMS Q/U and PI files 

In [ ]:
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QA_C_conv2.fits',QA_C_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QB_C_conv2.fits',QB_C_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QC_C_conv2.fits',QC_C_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QD_C_conv2.fits',QD_C_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UA_C_conv2.fits',UA_C_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UB_C_conv2.fits',UB_C_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UC_C_conv2.fits',UC_C_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UD_C_conv2.fits',UD_C_conv,header=hdr,overwrite=True)

fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QA_CG_conv2.fits',QA_GC_conv,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QB_CG_conv2.fits',QB_GC_conv,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QC_CG_conv2.fits',QC_GC_conv,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QD_CG_conv2.fits',QD_GC_conv,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UA_CG_conv2.fits',UA_GC_conv,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UB_CG_conv2.fits',UB_GC_conv,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UC_CG_conv2.fits',UC_GC_conv,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UD_CG_conv2.fits',UD_GC_conv,header=hdr,overwrite=True)


In [ ]:
PI_C_conv = np.nanmean(np.sqrt(np.array([C_Q_list])**2+np.array(C_U_list)**2),axis=0)
PI_CG_conv = np.nanmean(np.sqrt(np.array(CG_Q_list)**2+np.array(CG_U_list)**2),axis=0)

fits.writeto('/home2/DATA_AO/CGPS_GMIMS/PI_C.fits',PI_C,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/PI_G.fits',PI_G,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/PI_CG.fits',PI_CG,header=hdr,overwrite=True)

## Calculate polarisation angles

In [ ]:
raw_CG = False
if raw_CG:
    PA_A = 0.5*np.arctan2(CG_U_list[0],CG_Q_list[0])
    PA_B = 0.5*np.arctan2(CG_U_list[1],CG_Q_list[1])
    PA_C = 0.5*np.arctan2(CG_U_list[2],CG_Q_list[2])
    PA_D = 0.5*np.arctan2(CG_U_list[3],CG_Q_list[3])

conv_CG = False
if conv_CG:
    PA_A_conv = 0.5*np.arctan2(UA_GC_conv,QA_GC_conv)
    PA_B_conv = 0.5*np.arctan2(UB_GC_conv,QB_GC_conv)
    PA_C_conv = 0.5*np.arctan2(UC_GC_conv,QC_GC_conv)
    PA_D_conv = 0.5*np.arctan2(UD_GC_conv,QD_GC_conv)
    
conv_C = True
if conv_C:
    PA_A_conv = 0.5*np.arctan2(UA_C_conv,QA_C_conv)
    PA_B_conv = 0.5*np.arctan2(UB_C_conv,QB_C_conv)
    PA_C_conv = 0.5*np.arctan2(UC_C_conv,QC_C_conv)
    PA_D_conv = 0.5*np.arctan2(UD_C_conv,QD_C_conv)

G = False
if G:
    PA_A_G = 0.5*np.arctan2(G_U_list[0],G_Q_list[0])
    PA_B_G = 0.5*np.arctan2(G_U_list[1],G_Q_list[1])
    PA_C_G = 0.5*np.arctan2(G_U_list[2],G_Q_list[2])
    PA_D_G = 0.5*np.arctan2(G_U_list[3],G_Q_list[3])

In [ ]:
del CG_Q_list,CG_U_list,QA_GC_conv,QB_GC_conv,QC_GC_conv,QD_GC_conv,UB_GC_conv,UA_GC_conv,UC_GC_conv,UD_GC_conv
gc.collect()

In [ ]:
freq = np.array([1406.9,1413.8,1427.4,1434.3])
lbd2 = ((3e8)/(freq*1e6))**2
print(lbd2)

In [ ]:
if raw_CG:
    zz_A = np.exp(2*PA_A*1j)
    zz_B = np.exp(2*PA_B*1j)
    zz_C = np.exp(2*PA_C*1j)
    zz_D = np.exp(2*PA_D*1j)
    
if conv_CG:
    zz_A_conv = np.exp(2*PA_A_conv*1j)
    zz_B_conv = np.exp(2*PA_B_conv*1j)
    zz_C_conv = np.exp(2*PA_C_conv*1j)
    zz_D_conv = np.exp(2*PA_D_conv*1j)
    
if conv_C:
    zz_A_conv = np.exp(2*PA_A_conv*1j)
    zz_B_conv = np.exp(2*PA_B_conv*1j)
    zz_C_conv = np.exp(2*PA_C_conv*1j)
    zz_D_conv = np.exp(2*PA_D_conv*1j)
    
if G:
    zz_A_G = np.exp(2*PA_A_G*1j)
    zz_B_G = np.exp(2*PA_B_G*1j)
    zz_C_G = np.exp(2*PA_C_G*1j)
    zz_D_G = np.exp(2*PA_D_G*1j)

In [ ]:
if raw_CG:
    dtau_AB = np.arctan2((zz_B*np.conj(zz_A)).imag,(zz_B*np.conj(zz_A)).real)/2.
    dtau_BC = np.arctan2((zz_C*np.conj(zz_B)).imag,(zz_C*np.conj(zz_B)).real)/2.
    dtau_CD = np.arctan2((zz_D*np.conj(zz_C)).imag,(zz_D*np.conj(zz_C)).real)/2.

if conv_CG:
    dtau_AB_conv = np.arctan2((zz_B_conv*np.conj(zz_A_conv)).imag,(zz_B_conv*np.conj(zz_A_conv)).real)/2.
    dtau_BC_conv = np.arctan2((zz_C_conv*np.conj(zz_B_conv)).imag,(zz_C_conv*np.conj(zz_B_conv)).real)/2.
    dtau_CD_conv = np.arctan2((zz_D_conv*np.conj(zz_C_conv)).imag,(zz_D_conv*np.conj(zz_C_conv)).real)/2.

if conv_C:
    dtau_AB_conv = np.arctan2((zz_B_conv*np.conj(zz_A_conv)).imag,(zz_B_conv*np.conj(zz_A_conv)).real)/2.
    dtau_BC_conv = np.arctan2((zz_C_conv*np.conj(zz_B_conv)).imag,(zz_C_conv*np.conj(zz_B_conv)).real)/2.
    dtau_CD_conv = np.arctan2((zz_D_conv*np.conj(zz_C_conv)).imag,(zz_D_conv*np.conj(zz_C_conv)).real)/2.    
    
if G:
    dtau_AB_G = np.arctan2((zz_B_G*np.conj(zz_A_G)).imag,(zz_B_G*np.conj(zz_A_G)).real)/2.
    dtau_BC_G = np.arctan2((zz_C_G*np.conj(zz_B_G)).imag,(zz_C_G*np.conj(zz_B_G)).real)/2.
    dtau_CD_G = np.arctan2((zz_D_G*np.conj(zz_C_G)).imag,(zz_D_G*np.conj(zz_C_G)).real)/2.

In [ ]:
if raw_CG:
    PA_A_fix = PA_A.copy()
    PA_B_fix = PA_A_fix + dtau_AB
    PA_C_fix = PA_B_fix + dtau_BC
    PA_D_fix = PA_C_fix + dtau_CD
if conv_CG:
    PA_A_fix_conv = PA_A_conv.copy()
    PA_B_fix_conv = PA_A_fix_conv + dtau_AB_conv
    PA_C_fix_conv = PA_B_fix_conv + dtau_BC_conv
    PA_D_fix_conv = PA_C_fix_conv + dtau_CD_conv
if conv_C:
    PA_A_fix_conv = PA_A_conv.copy()
    PA_B_fix_conv = PA_A_fix_conv + dtau_AB_conv
    PA_C_fix_conv = PA_B_fix_conv + dtau_BC_conv
    PA_D_fix_conv = PA_C_fix_conv + dtau_CD_conv
if G:
    PA_A_fix_G = PA_A_G.copy()
    PA_B_fix_G = PA_A_fix_G + dtau_AB_G
    PA_C_fix_G = PA_B_fix_G + dtau_BC_G
    PA_D_fix_G = PA_C_fix_G + dtau_CD_G

In [ ]:
if raw_CG:
    del zz_A,zz_B,zz_C,zz_D
    del dtau_AB,dtau_BC,dtau_CD
if conv_CG:
    del zz_A_conv,zz_B_conv,zz_C_conv,zz_D_conv
    del dtau_AB_conv,dtau_BC_conv,dtau_CD_conv
if conv_C:
    del zz_A_conv,zz_B_conv,zz_C_conv,zz_D_conv
    del dtau_AB_conv,dtau_BC_conv,dtau_CD_conv
if G:
    del zz_A_G,zz_B_G,zz_C_G,zz_D_G
    del dtau_AB_G,dtau_BC_G,dtau_CD_G
gc.collect()

## Calculate RMs for original and convolved Q/U data

In [ ]:
%%time
RM_GC_Anna = np.empty_like(PA_A_fix)
RM_GC_Anna_conv = np.empty_like(PA_A_fix)

for i in range(0,PA_A_fix.shape[0]):
    print(i)
    for j in range(0,PA_A_fix.shape[1]):
        if np.isfinite(PA_A_fix[i,j]):
            PA_fix = np.array([PA_A_fix[i,j],PA_B_fix[i,j],PA_C_fix[i,j],PA_D_fix[i,j]])
            try:
                RM_GC_Anna[i,j] = np.polyfit(lbd2,PA_fix,1)[0]
            except:
                pass
            del PA_fix
        if np.isfinite(PA_A_fix_conv[i,j]):
            PA_fix = np.array([PA_A_fix_conv[i,j],PA_B_fix_conv[i,j],PA_C_fix_conv[i,j],PA_D_fix_conv[i,j]])
            try:
                RM_GC_Anna_conv[i,j] = np.polyfit(lbd2,PA_fix,1)[0]
            except:
                pass
            del PA_fix
    gc.collect()

In [ ]:
%%time
RM_G_Anna = np.empty_like(PA_A_fix_G)

for i in range(0,PA_A_fix_G.shape[0]):
    print(i)
    for j in range(0,PA_A_fix_G.shape[1]):
        if np.isfinite(PA_A_fix_G[i,j]):
            PA_fix = np.array([PA_A_fix_G[i,j],PA_B_fix_G[i,j],PA_C_fix_G[i,j],PA_D_fix_G[i,j]])
            try:
                RM_G_Anna[i,j] = np.polyfit(lbd2,PA_fix,1)[0]
            except:
                pass
            del PA_fix
    gc.collect()

In [ ]:
%%time
RM_C_Anna_conv = np.empty_like(PA_A_fix_conv)

for i in range(0,PA_A_fix_conv.shape[0]):
    print(i)
    for j in range(0,PA_A_fix_conv.shape[1]):
        if np.isfinite(PA_A_fix_conv[i,j]):
            PA_fix = np.array([PA_A_fix_conv[i,j],PA_B_fix_conv[i,j],PA_C_fix_conv[i,j],PA_D_fix_conv[i,j]])
            try:
                RM_C_Anna_conv[i,j] = np.polyfit(lbd2,PA_fix,1)[0]
            except:
                pass
            del PA_fix
    gc.collect()

## Write out FITS files for RM

In [ ]:
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/CG_RM_AO.fits',RM_GC_Anna,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/CG_RM_AO_conv_3.fits',RM_GC_Anna_conv,header=hdr,overwrite=True)
#fits.writeto('/home2/DATA_AO/CGPS_GMIMS/G_RM_AO.fits',RM_G_Anna,header=hdr,overwrite=True)
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/C_RM_AO_conv_2.fits',RM_C_Anna_conv,header=hdr,overwrite=True)

In [ ]:
wcs = WCS(hdr_new)
#plt.figure(figsize=(120,20))
plt.figure(figsize=(20,10))
plt.subplot(projection=wcs)
plt.imshow(PA_A*180/np.pi,origin='lower',vmin=-90,vmax=90,cmap='twilight')
#plt.grid(color='white', ls='solid')
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')
#plt.savefig('/home/ordoga/Python/CGPS_GMIMS_PLOTS/PA_A.pdf')

In [ ]:
GMIMS_aq = fits.open('/home/ordoga/DRAO_export/CG_W23/GMIMS_a/QA_GMIMS_W21_HR.fits')
print(GMIMS_aq[0].data.shape)
plt.imshow(GMIMS_aq[0].data,origin='lower',vmin=-0.2,vmax=0.2,cmap='RdBu')

In [ ]:
idxs = np.where((PF_GC>0.5) & (RM_GC>1500))
j = idxs[0][10]
i = idxs[1][10]
print(j)
print(i)

print(np.arctan2(zz_A.imag[j,i],zz_A.real[j,i])/2)
print(PA_A[j,i])
#print(zz_A[j,i],np.conj(zz_A[j,i]))
print(dtau_AB[j,i]*180/np.pi)
print((PA_B[j,i]-PA_A[j,i])*180/np.pi)

In [ ]:

PA = np.array([PA_A[j,i],PA_B[j,i],PA_C[j,i],PA_D[j,i]])*180./np.pi
PA_fix = np.array([PA_A_fix[j,i],PA_B_fix[j,i],PA_C_fix[j,i],PA_D_fix[j,i]])*180./np.pi
print(PA)
plt.scatter(lbd2,PA,s=100)
plt.scatter(lbd2,PA_fix,s=50,color='k')
plt.ylim(-360,360)
plt.grid()

In [ ]:
idxs = np.where((PF_GC>0.5) & (RM_GC>1500))
jj = idxs[0][10]
ii = idxs[1][10]
print(jj)
print(ii)

fig,axs = plt.subplots(5,5,figsize=(15,15))
for i in range(0,5):
    for j in range(0,5):
        PA = np.array([PA_A[j+jj,i+ii],PA_B[j+jj,i+ii],PA_C[j+jj,i+ii],PA_D[j+jj,i+ii]])*180./np.pi
        PA_fix = np.array([PA_A_fix[j+jj,i+ii],PA_B_fix[j+jj,i+ii],PA_C_fix[j+jj,i+ii],PA_D_fix[j+jj,i+ii]])*180./np.pi
        coef = np.polyfit(lbd2,PA_fix,1)
        #print(coef)
        poly1d_fn = np.poly1d(coef)      
        print(PA)
        axs[i,j].scatter(lbd2,PA,s=100)
        axs[i,j].scatter(lbd2,PA_fix,s=50,color='k')
        axs[i,j].plot(lbd2,poly1d_fn(lbd2))
        axs[i,j].set_ylim(-360,360)
        axs[i,j].grid()

In [ ]:
print(RM_GC.shape)

In [ ]:
RM_GC_Anna = np.empty_like(RM_GC)

for i in range(0,RM_GC.shape[0]):
#for i in range(1000,1010):
    print(i)
    for j in range(0,RM_GC.shape[1]):
    #for j in range(0,1000):
        if np.isfinite(PA_A_fix[i,j]):
            PA_fix = np.array([PA_A_fix[i,j],PA_B_fix[i,j],PA_C_fix[i,j],PA_D_fix[i,j]])
            #coef = np.polyfit(lbd2,PA_fix,1)
            #poly1d_fn = np.poly1d(coef)
            try:
                RM_GC_Anna[i,j] = np.polyfit(lbd2,PA_fix,1)[0]
            except:
                pass

In [ ]:
fits.writeto('../CGPS_GMIMS_PLOTS/QA_conv.fits',QA_GC_conv,header=hdr_new,overwrite=True)
fits.writeto('../CGPS_GMIMS_PLOTS/QB_conv.fits',QB_GC_conv,header=hdr_new,overwrite=True)
fits.writeto('../CGPS_GMIMS_PLOTS/QC_conv.fits',QC_GC_conv,header=hdr_new,overwrite=True)
fits.writeto('../CGPS_GMIMS_PLOTS/QD_conv.fits',QD_GC_conv,header=hdr_new,overwrite=True)
fits.writeto('../CGPS_GMIMS_PLOTS/UA_conv.fits',UA_GC_conv,header=hdr_new,overwrite=True)
fits.writeto('../CGPS_GMIMS_PLOTS/UB_conv.fits',UB_GC_conv,header=hdr_new,overwrite=True)
fits.writeto('../CGPS_GMIMS_PLOTS/UC_conv.fits',UC_GC_conv,header=hdr_new,overwrite=True)
fits.writeto('../CGPS_GMIMS_PLOTS/UD_conv.fits',UD_GC_conv,header=hdr_new,overwrite=True)

In [ ]:
fits.writeto('../CGPS_GMIMS_PLOTS/RM_G_JCB.fits',RM_G,header=hdr_new,overwrite=True)


In [ ]:
wcs = WCS(hdr_new)
plt.figure(figsize=(80,20))
plt.subplot(projection=wcs)
plt.imshow(RM_GC_Anna,origin='lower',vmin=-200,vmax=200,cmap='bwr')
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')
plt.savefig('/home/ordoga/Python/CGPS_GMIMS_PLOTS/RM_GMIMS_CGPS_AO.pdf')

In [ ]:
lbd2_arr = np.linspace(lbd2[0],lbd2[-1],1000)


In [ ]:
idxs = np.where(abs(RM_GC_Anna-RM_GC)>1000)
jj = idxs[0][5000]
ii = idxs[1][5000]
print(jj)
print(ii)

fig,axs = plt.subplots(3,3,figsize=(15,15))
for i in range(0,3):
    for j in range(0,3):
        PA = np.array([PA_A[j+jj,i+ii],PA_B[j+jj,i+ii],PA_C[j+jj,i+ii],PA_D[j+jj,i+ii]])*180./np.pi
        PA_fix = np.array([PA_A_fix[j+jj,i+ii],PA_B_fix[j+jj,i+ii],PA_C_fix[j+jj,i+ii],PA_D_fix[j+jj,i+ii]])*180./np.pi
        axs[i,j].scatter(lbd2,PA,s=100)
        axs[i,j].scatter(lbd2,PA_fix,s=50,color='k')
        axs[i,j].plot(lbd2_arr,RM_GC[j+jj,i+ii]*(lbd2_arr-lbd2_arr[500])*180./np.pi,color='red')
        axs[i,j].plot(lbd2_arr,RM_GC_Anna[j+jj,i+ii]*(lbd2_arr-lbd2_arr[500])*180./np.pi,color='k')
        axs[i,j].text(lbd2[0]-0.0002,270,str(round(RM_GC[j+jj,i+ii],1)))
        axs[i,j].text(lbd2[0]-0.0002,230,str(round(RM_GC_Anna[j+jj,i+ii],1)))
        axs[i,j].set_ylim(-300,300)
        axs[i,j].grid()